## Adam Algorithm in Theory

The algorithm to optimize a stochastic objective function $f(\theta)$ with parameters $\theta$ with the Adam method can be summarized verbally as follows:

until the parameters have converged, repeat these steps:

    - compute the gradients w.r.t. the stochastic objective at the last timestep

    - update the mean estimate

    - update the raw variance estimate

    - bias-correct these estimates
    
    - update the parameters

## Adam Algorithm in Practice

### Imports and Setup

In [3]:
import math
import numpy as np
import torch

from typing import Callable

### Class Definition

In [4]:
class Adam():

    def __init__(
            self, 
            func: Callable[[torch.Tensor], float], 
            alpha: float = 0.001, 
            beta_1: float = 0.9, 
            beta_2: float = 0.999, 
            epsilon: float = 10e-8, 
            max_iters: int = 10000,
            grad_tol: float = 1e-4
        ) -> None:
        self.alpha = alpha
        self.beta_1 = beta_1
        self.beta_2 = beta_2
        self.epsilon = epsilon
        self.func = func
        self.max_iters = max_iters
        self.grad_tol = grad_tol
        return


    def step(self, t: int, current_theta: torch.Tensor, m: torch.Tensor, v: torch.Tensor) -> (torch.Tensor, torch.Tensor, torch.Tensor, float):

        # zero the gradients
        current_theta.grad = None

        # compute the gradients of f w.r.t. the current parameters
        # forward pass
        objective = self.func(current_theta)
        # backward pass
        objective.backward()
        g = current_theta.grad
        g_norm = g.max().item()
    
        # compute stochastic moments
        m = self.beta_1 * m + (1 - self.beta_1) * g
        v = self.beta_2 * v + (1 - self.beta_2) * g ** 2
        a = self.alpha * (math.sqrt(1 - self.beta_2 ** t)/(1 - self.beta_1 ** t))

        # update parameters
        with torch.no_grad():
            next_theta = current_theta - a * m / (torch.sqrt(v) + self.epsilon)

        return next_theta, m, v, g_norm
        
        
    def optimize(self, theta: np.ndarray) -> np.ndarray:

        m = torch.zeros(theta.shape, dtype=torch.float)
        v = torch.zeros(theta.shape, dtype=torch.float)

        theta = np.asarray(theta, dtype=np.float32)
        last_theta = torch.from_numpy(theta)
        next_theta = torch.from_numpy(theta) + 2 * self.epsilon

        t = 0
        g_norm = 1.0

        # until the parameters converge
        while g_norm > self.grad_tol:

            t += 1
            last_theta = next_theta.detach().requires_grad_(True)
            next_theta, m, v, g_norm = self.step(t, last_theta, m, v)

            if t > self.max_iters:
                break

        return next_theta.detach().numpy()

## Test

### Test Models

In [5]:
class Test1D:

    def __init__(self) -> None:
        return

    @staticmethod
    def func(theta: torch.Tensor) -> float:
        if theta.shape[0] != 2:
            raise ValueError

        target = torch.tensor([1,-2], dtype=theta.dtype)
        loss = ((theta - target) ** 2).mean()

        return loss

In [6]:
class TestND:

    def __init__(self, shape: list[int]) -> None:
        self.shape = shape
        self.target = torch.randn(*shape, dtype=torch.float)
        return

    def func(self, theta: torch.Tensor) -> float:
        loss = ((theta - self.target) ** 2).mean()

        return loss

### Optimization

In [7]:
test = Test1D()
adam = Adam(test.func, max_iters=1000000, grad_tol=1e-8)

initial_theta = np.random.rand(2,)

optimal_theta = adam.optimize(initial_theta)

print(optimal_theta)

[ 1. -2.]


In [9]:
from torch import allclose


shape = [4,3,2]
test = TestND(shape=shape)
adam = Adam(test.func, max_iters=1000000, grad_tol=1e-8)

initial_theta = np.random.rand(*shape)

optimal_theta = adam.optimize(initial_theta)

print(allclose(((test.target - optimal_theta) ** 2).mean(),torch.zeros_like(test.target.mean())))

True


/var/folders/xk/wh_dv4fd785263srbm07380m0000gn/T/ipykernel_11258/3575701905.py:12: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  print(allclose(((test.target - optimal_theta) ** 2).mean(),torch.zeros_like(test.target.mean())))


The optimizer appears to correctly optimize the parameters to minimize simple loss functions and passes the initial vibe check.